# 07. 시간대별 UTCI 열스트레스 지수 — deliverable 1 (물리 기반)

위성 대리 지수 → **물리 기반 인체 열지수**로 도약. 폭염일 대표 24시간.

`SVF(건물) + LST(지표) + NDVI(나무그늘) + 기상24h → Tmrt(복사) → UTCI(인체)`

- 격자별 24h UTCI → **daytime 평균·피크·강한스트레스 시간수**
- 공간변이는 복사(SVF·LST·그늘)에서, 시간변이는 태양고도+기상에서.

산출물: `data/results/heat_index_utci.geojson`, `output/figures/fig10_utci.png`

In [1]:
# ── 실행 환경 가드 (하이브리드 전략, 실행순서.md §1 참조) ──────────────
# 이 노트북은 '로컬 사전계산' 단계입니다. COMPAS에는 ['pythermalcomfort'] 이(가) 없으므로
# 그 경우 아래 셀들은 자동으로 건너뛰고(LOCAL_RUN=False) 동봉 산출물만 검증합니다.
# 산출물은 제출 데이터 ZIP에 포함되어 있어 후속 노트북은 그대로 실행됩니다.
import importlib.util, pathlib
_DEPS = ['pythermalcomfort']
_missing = [m for m in _DEPS if importlib.util.find_spec(m) is None]
LOCAL_RUN = not _missing
_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
if LOCAL_RUN:
    print("[로컬 모드] 의존성 충족 - 전체 셀을 실행합니다.")
else:
    print("[COMPAS 모드] 미설치 의존성:", _missing, "-> 아래 셀은 건너뜁니다.")
    print("이 단계의 동봉 산출물 검증:")
    _o = _ROOT / "data/results/heat_index_utci.geojson"
    _h = _ROOT / "data/local/grid_heat.csv.gz"
    if _o.exists():
        print("  [OK] data/results/heat_index_utci.geojson (%.1f MB)" % (_o.stat().st_size / 1e6))
    elif _h.exists():
        import geopandas as _gpd, pandas as _pd, glob as _glob
        _hd = _pd.read_csv(_h)
        _g4 = _gpd.read_file(sorted(_glob.glob(str(_ROOT / "data/raw/04*")))[0])[["grid_id", "geometry"]]
        _g4["grid_id"] = _g4["grid_id"].astype(int)
        _cols = [x for x in ["grid_id","zoneName","blockType","svf","utci_daymean","utci_peak","utci_strong_hrs","peak_lowconf"] if x in _hd.columns]
        _out = _g4.merge(_hd[_cols], on="grid_id", how="inner")
        _o.parent.mkdir(parents=True, exist_ok=True)
        _out.to_file(_o, driver="GeoJSON", COORDINATE_PRECISION=6)
        print("  [복원] grid_heat.csv.gz + 04격자 -> heat_index_utci.geojson (%d행)" % len(_out))
    else:
        print("  [!!] heat_index_utci.geojson / grid_heat.csv.gz 둘 다 없음 - 실행순서.md 0단계 참조")


[로컬 모드] 의존성 충족 - 전체 셀을 실행합니다.


In [2]:
if LOCAL_RUN:
    import sys, pathlib
    sys.path.append(str(pathlib.Path.cwd().parent))
    import numpy as np, pandas as pd, geopandas as gpd
    from pythermalcomfort.models import utci
    from src.config import DATA_PROCESSED, DATA_RESULTS, DATA_EXTERNAL, OUT_FIGURES, CRS_LANDSAT, CRS_SUBMIT
    from src.microclimate.svf import compute_svf
    from src.microclimate.mrt import solar_elevation, clearsky, tmrt

    g = gpd.read_file(DATA_PROCESSED / "grid_features.geojson").to_crs(CRS_LANDSAT)
    # SVF
    b = gpd.read_file(DATA_EXTERNAL / "buildings_3d" / "buildings.geojson").to_crs(CRS_LANDSAT)
    h = pd.to_numeric(b["height"], errors="coerce"); gf = pd.to_numeric(b["grnd_flr"], errors="coerce")
    b["h"] = np.where(h.isna() | (h <= 0), gf * 3.3, h); b["h"] = pd.to_numeric(b["h"], errors="coerce").clip(0).fillna(0)
    gc = np.c_[g.geometry.centroid.x, g.geometry.centroid.y]
    g["svf"] = compute_svf(gc, np.c_[b.geometry.centroid.x, b.geometry.centroid.y], b["h"].values)
    lst_peak = g["lst_peak"].fillna(g["lst_peak"].median()).values
    ndvi = g["ndvi"].fillna(0).values
    svf = g["svf"].values
    print("격자", len(g), "| SVF 중위", round(float(np.median(svf)), 3))

격자 54029 | SVF 중위 0.988


In [3]:
if LOCAL_RUN:
    # 폭염일 대표 24시간 (수원, 일최고>33 인 날들의 시간별 평균)
    w = pd.read_csv(DATA_PROCESSED / "weather_hourly.csv")
    w = w[w["stn_name"].astype(str).str.contains("수원")].copy()
    w["tm"] = pd.to_datetime(w["tm"]); w["date"] = w["tm"].dt.date
    hot = w.groupby("date")["ta"].max()
    hot_days = hot[hot > 33].index
    if len(hot_days) < 3:
        hot_days = hot.nlargest(10).index
    wh = w[w["date"].isin(hot_days)].groupby("hour")[["ta", "hm", "ws"]].mean()
    print(f"폭염일 {len(hot_days)}일. 대표 24h 기온: {wh['ta'].min():.1f}~{wh['ta'].max():.1f}°C (피크 {wh['ta'].idxmax()}시)")

폭염일 17일. 대표 24h 기온: 25.8~33.8°C (피크 15시)


In [4]:
if LOCAL_RUN:
    # 24시간 루프: 태양고도→복사→Tmrt→UTCI (격자 벡터)
    LAT, DOY = 37.4, 218   # 8월초
    Ta15 = wh.loc[15, "ta"]
    el15 = solar_elevation(LAT, DOY, 15)
    dTs_peak = lst_peak - Ta15                       # 지표 과열(피크)
    s15 = np.sin(np.radians(el15))

    UT = np.full((24, len(g)), np.nan, dtype=np.float32)
    for hr in range(24):
        Ta, RH, WS = wh.loc[hr, "ta"], wh.loc[hr, "hm"], max(float(wh.loc[hr, "ws"]), 0.5)
        el = solar_elevation(LAT, DOY, hr)
        kglob, kdif, kdir = clearsky(el)
        sfac = np.clip(np.sin(np.radians(el)) / s15, 0, None) if el > 0 else 0.0
        Ts = Ta + dTs_peak * sfac                     # 격자별 시간별 지표온도
        Tmrt = tmrt(Ta, Ts, svf, ndvi, el, kglob, kdif, kdir)
        UT[hr] = utci(tdb=Ta, tr=np.clip(Tmrt, Ta - 30, Ta + 70), v=min(WS, 17), rh=RH).utci
    print("UTCI 계산 완료. 15시 범위:", round(float(np.nanmin(UT[15])), 1), "~", round(float(np.nanmax(UT[15])), 1))

UTCI 계산 완료. 15시 범위: 35.8 ~ 38.4


In [5]:
if LOCAL_RUN:
    # 24h → 지수: 주간평균·피크·강한열스트레스(>32) 시간수
    gd = g.copy()
    gd["utci_daymean"] = UT[9:19].mean(axis=0)
    gd["utci_peak"] = UT.max(axis=0)
    gd["utci_strong_hrs"] = (UT > 32).sum(axis=0)
    print("지구별 (주간평균 UTCI / 피크 / 강스트레스시간):")
    print(gd.groupby("zoneName")[["utci_daymean", "utci_peak", "utci_strong_hrs"]].mean().round(1))
    # 검증: 양지(고SVF) vs 음지(저SVF) 15시 UTCI
    q = gd["svf"]
    print("\n[검증] 15시 UTCI  개방(SVF>0.98):", round(float(UT[15][q > 0.98].mean()), 1),
          "vs 폐색(SVF<0.85):", round(float(UT[15][q < 0.85].mean()), 1))
    print("[검증] 새벽3시 평균 UTCI", round(float(np.nanmean(UT[3])), 1), "(기온", round(float(wh.loc[3, 'ta']), 1), "°C)")

지구별 (주간평균 UTCI / 피크 / 강스트레스시간):
               utci_daymean  utci_peak  utci_strong_hrs
zoneName                                               
고양창릉 공공주택지구       34.900002  37.000000              9.3
남양주왕숙 공공주택지구      34.900002  37.099998              9.3
성남분당              35.099998  37.400002              9.0
성남판교지구 택지개발사업     35.000000  37.200001              8.8

[검증] 15시 UTCI  개방(SVF>0.98): 36.6 vs 폐색(SVF<0.85): 37.4
[검증] 새벽3시 평균 UTCI 26.3 (기온 26.1 °C)


In [6]:
if LOCAL_RUN:
    DATA_RESULTS.mkdir(parents=True, exist_ok=True)
    cols = ["grid_id", "zoneName", "blockType", "svf", "utci_daymean", "utci_peak", "utci_strong_hrs", "peak_lowconf", "geometry"]
    gd[cols].to_crs(CRS_SUBMIT).to_file(DATA_RESULTS / "heat_index_utci.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

    import matplotlib.pyplot as plt
    from src.viz.fonts import setup_font
    setup_font()   # 한글 폰트 자동 적용(COMPAS 미보유 시 동봉 폰트 등록)
    fig, ax = plt.subplots(1, 3, figsize=(21, 7))
    gd.plot(column="utci_daymean", cmap="turbo", ax=ax[0], linewidth=0, legend=True, legend_kwds={"shrink": 0.5})
    ax[0].set_title("주간평균 UTCI (°C)")
    gd.to_crs(CRS_LANDSAT).assign(u=UT[15]).plot(column="u", cmap="turbo", ax=ax[1], linewidth=0, legend=True, legend_kwds={"shrink": 0.5})
    ax[1].set_title("15시 UTCI (폭염 피크)")
    gd.plot(column="utci_strong_hrs", cmap="YlOrRd", ax=ax[2], linewidth=0, legend=True, legend_kwds={"shrink": 0.5})
    ax[2].set_title("강한 열스트레스 시간수 (>32°C)")
    for a in ax: a.set_axis_off()
    plt.tight_layout(); plt.savefig(OUT_FIGURES / "fig10_utci.png", dpi=120); plt.close()

In [7]:
if LOCAL_RUN:
    # 기상 지점 민감도: 수원 대신 서울 ASOS로 강제력 교체 시 공간 순위가 유지되는가
    ws = pd.read_csv(DATA_PROCESSED / "weather_hourly.csv")
    ws = ws[ws["stn_name"].astype(str).str.contains("서울")].copy()
    ws["tm"] = pd.to_datetime(ws["tm"]); ws["date"] = ws["tm"].dt.date
    hot_s = ws.groupby("date")["ta"].max()
    hd = hot_s[hot_s > 33].index
    if len(hd) < 3: hd = hot_s.nlargest(10).index
    whs = ws[ws["date"].isin(hd)].groupby("hour")[["ta", "hm", "ws"]].mean()
    Ta15s = whs.loc[15, "ta"]; dTs_s = lst_peak - Ta15s
    UTs = np.full((24, len(g)), np.nan, dtype=np.float32)
    for hr in range(24):
        Ta, RH, WS = whs.loc[hr, "ta"], whs.loc[hr, "hm"], max(float(whs.loc[hr, "ws"]), 0.5)
        el = solar_elevation(LAT, DOY, hr)
        kglob, kdif, kdir = clearsky(el)
        sfac = np.clip(np.sin(np.radians(el)) / s15, 0, None) if el > 0 else 0.0
        Ts = Ta + dTs_s * sfac
        Tm = tmrt(Ta, Ts, svf, ndvi, el, kglob, kdif, kdir)
        UTs[hr] = utci(tdb=Ta, tr=np.clip(Tm, Ta - 30, Ta + 70), v=min(WS, 17), rh=RH).utci
    dm_s = UTs[9:19].mean(axis=0)
    from scipy.stats import spearmanr
    rho_st = spearmanr(gd["utci_daymean"], dm_s).statistic
    a = gd["utci_daymean"].rank(pct=True) >= 0.9; b = pd.Series(dm_s).rank(pct=True) >= 0.9
    jac = (a & b).sum() / (a | b).sum()
    print("기상 지점 민감도(수원->서울 강제력 교체): 폭염일 %d일" % len(hd))
    print("  utci_daymean 순위상관 rho=%.4f | 상위10%% Jaccard=%.2f" % (rho_st, jac))

기상 지점 민감도(수원->서울 강제력 교체): 폭염일 22일
  utci_daymean 순위상관 rho=0.9997 | 상위10% Jaccard=0.96
